In [ ]:
# scripts/4_train_unet.py
"""
Train a multi-channel U-Net with two heads:
 - head_reg: predicts height map (L1)
 - head_mask: predicts built/veg mask (BCE)
Loss = L1_height + alpha * BCE_mask + beta * edge_loss
"""
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.optim import AdamW
from tqdm import tqdm
from scripts.dataset_unet import GeoPatchDataset  # assume module placed accordingly
import numpy as np
import torch.nn.functional as F
import math

# ---------------------------
# Simple U-Net implementation (adapted, lightweight)
# ---------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class UNetMulti(nn.Module):
    def __init__(self, in_channels=5, base_filters=32, out_mask_ch=2):
        super().__init__()
        f = base_filters
        self.enc1 = DoubleConv(in_channels, f)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(f, f*2)
        self.enc3 = DoubleConv(f*2, f*4)
        self.enc4 = DoubleConv(f*4, f*8)
        self.center = DoubleConv(f*8, f*16)
        # decoder
        self.up4 = nn.ConvTranspose2d(f*16, f*8, 2, stride=2)
        self.dec4 = DoubleConv(f*16, f*8)
        self.up3 = nn.ConvTranspose2d(f*8, f*4, 2, stride=2)
        self.dec3 = DoubleConv(f*8, f*4)
        self.up2 = nn.ConvTranspose2d(f*4, f*2, 2, stride=2)
        self.dec2 = DoubleConv(f*4, f*2)
        self.up1 = nn.ConvTranspose2d(f*2, f, 2, stride=2)
        self.dec1 = DoubleConv(f*2, f)
        # heads
        self.head_reg = nn.Conv2d(f, 1, kernel_size=1)     # height regression
        self.head_mask = nn.Conv2d(f, out_mask_ch, kernel_size=1)  # built/veg multi-label

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        c = self.center(self.pool(e4))
        d4 = self.up4(c)
        d4 = torch.cat([d4, e4], dim=1); d4 = self.dec4(d4)
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1); d3 = self.dec3(d3)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1); d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1); d1 = self.dec1(d1)
        reg = self.head_reg(d1)
        mask_logits = self.head_mask(d1)
        return reg, mask_logits

# ---------------------------
# Edge loss helper (Sobel)
# ---------------------------
def edge_loss(pred, target):
    # pred, target: Bx1xHxW
    sobel_x = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=torch.float32, device=pred.device).unsqueeze(0).unsqueeze(0)
    sobel_y = torch.tensor([[1,2,1],[0,0,0],[-1,-2,-1]], dtype=torch.float32, device=pred.device).unsqueeze(0).unsqueeze(0)
    px = F.conv2d(pred, sobel_x, padding=1)
    py = F.conv2d(pred, sobel_y, padding=1)
    tx = F.conv2d(target, sobel_x, padding=1)
    ty = F.conv2d(target, sobel_y, padding=1)
    return F.l1_loss(px, tx) + F.l1_loss(py, ty)

# ---------------------------
# Training loop
# ---------------------------
if __name__ == "__main__":
    processed_dir = "../data/processed"
    # build tile list by scanning processed dir for DSM files
    tiles = []
    for p in os.listdir(processed_dir):
        if p.endswith("_DSM.tif"):
            tiles.append(p.replace("_DSM.tif",""))
    # spatial holdout: keep last 20% tiles as val (naïf). Better: use geo split by region.
    split = int(len(tiles)*0.8)
    train_tiles = tiles[:split]
    val_tiles = tiles[split:]

    train_ds = GeoPatchDataset(processed_dir, train_tiles, patch_size=256)
    val_ds = GeoPatchDataset(processed_dir, val_tiles, patch_size=256)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UNetMulti(in_channels=5, base_filters=32, out_mask_ch=2).to(device)

    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs = 60
    alpha = 1.0  # weight for BCE mask
    beta = 0.5   # weight for edge_loss

    best_val_loss = 1e9
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for xb, yh, ym in pbar:
            xb = xb.to(device)
            yh = yh.to(device)       # regression target
            ym = ym.to(device)       # mask target (2 channels)
            optimizer.zero_grad()
            pred_h, pred_mask_logits = model(xb)
            # regression loss L1 (MAE)
            loss_reg = F.l1_loss(pred_h, yh)
            # BCE mask (use logits + BCEWithLogits)
            loss_mask = F.binary_cross_entropy_with_logits(pred_mask_logits, ym)
            # edge loss between pred_h and yh
            loss_edge = edge_loss(pred_h, yh)
            loss = loss_reg + alpha*loss_mask + beta*loss_edge
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix({'loss': running_loss / (pbar.n+1)})
        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yh, ym in val_loader:
                xb = xb.to(device); yh = yh.to(device); ym = ym.to(device)
                pred_h, pred_mask_logits = model(xb)
                loss_reg = F.l1_loss(pred_h, yh).item()
                loss_mask = F.binary_cross_entropy_with_logits(pred_mask_logits, ym).item()
                loss_edge = edge_loss(pred_h, yh).item()
                val_loss = loss_reg + alpha*loss_mask + beta*loss_edge
                val_losses.append(val_loss)
        avg_val = np.mean(val_losses)
        print(f"Epoch {epoch+1} val_loss: {avg_val:.4f}")
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), "../outputs/unet_multitask_best.pth")
            print("Saved best model")
